# 2. Data Preprocessing and Artifact Generation

This notebook is designed to transform the raw data into a clean, structured format suitable for training machine learning models.

**Steps:**
1.  **Data Filtering:** Drop useless columns and rows with missing target values. Additionally, constrain the target variable (`price`) to a specific range to filter out extreme outliers.
2.  **Feature Engineering:** Derive meaningful features from unstructured data, such as parsing numerical values from text, counting items in lists, and calculating temporal metrics.
3.  **Feature Selection:** Identify and separate numerical and categorical input variables.
4.  **Data Imputation:** Handle missing values using a hybrid strategy: manual imputation for logical zeros and statistical imputation (median for numerics, 'unknown' for categoricals) within the pipeline.
5.  **Data Transformation:** Apply `StandardScaler` to normalize numerical features and `OneHotEncoder` to convert categorical variables into a machine-readable format using a `ColumnTransformer`.
6.  **Artifacts Export:** Serialize the processed dataset and the fitted preprocessor object (using pickle) to ensure reproducibility during inference.

In [ ]:
import pandas as pd
import pickle
import os
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from datetime import datetime

## Data Loading

In [ ]:
pd.set_option('display.max_columns', None)
df = pd.read_csv('../../data/madrid/listings.csv')
print(f"Original shape: {df.shape}")

## Data Filtering

In [ ]:
# Drop IDs, URLs, and scrape metadata (no predictive value)
df = df.drop(['id', 'host_id', 'scrape_id', 'listing_url', 'picture_url', 'host_url', 'host_thumbnail_url', 'host_picture_url',
              'last_scraped', 'source', 'calendar_last_scraped'], axis=1, errors='ignore')

# Drop high cardinality and unstructured text
df = df.drop(['name', 'description', 'host_name', 'host_verifications'], axis=1, errors='ignore')

# Drop high missing values
df = df.drop(['calendar_updated', 'license', 'host_response_rate', 'host_acceptance_rate', 'neighborhood_overview',
              'host_neighbourhood', 'neighbourhood', 'host_location', 'host_about'], axis=1, errors='ignore')


# Drop potential data leakage columns
# These metrics are often calculated post-hoc using the target variable itself.
# Mathematically, Estimated Revenue ≈ Price * Occupancy * Nights.
# If included, the model learns to algebraically reverse-engineer the 'price' from the 'revenue'
# (Price ≈ Revenue / Factor) rather than learning the market valuation of the property characteristics.
df = df.drop(['estimated_occupancy_l365d', 'estimated_revenue_l365d'], axis=1, errors='ignore')

# Drop redundant columns
# - Dates (processed into numerical 'days_since')
# - 'bathrooms_text' (redundant with 'bathrooms')
# - Nights logic (redundant with 'minimum_nights_avg_ntm' and 'maximum_nights_avg_ntm')
# - Host counts (redundant with 'calculated_host_listings_count')
# - Number of reviews (redundant with 'reviews_per_month' and 'number_of_reviews')
# - 'availability_eoy'(redundant with 'availability_30', 'availability_60', 'availability_90' and 'availability_365')
# - 'has_availability', 'host_has_profile_pic' (low variance booleans)
df = df.drop(['bathrooms_text',
              'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 
              'minimum_maximum_nights', 'maximum_maximum_nights',
              'host_listings_count', 'host_total_listings_count', 
              'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 
              'calculated_host_listings_count_shared_rooms',
              'number_of_reviews_ltm', 'number_of_reviews_l30d', 'number_of_reviews_ly',
              'availability_eoy',
              'has_availability', 'host_has_profile_pic'], axis=1, errors='ignore')

# Drop rows where target is missing
df = df.dropna(subset=['price'])

# Clean characters ('$') and convert to float if necessary
if df['price'].dtype == 'object':
    df['price'] = df['price'].replace('[\$,]', '', regex=True).astype(float)

# Filter prices between €0 and €249
df = df[(df['price'] > 0) & (df['price'] <= 249)]

## Feature Engineering

In [ ]:
def count_items(text):
    # Counts items in string lists like '["Wifi", "AC"]'
    if pd.isna(text): return 0
    return len(str(text).split(','))

def days_since(date_str):
    # Calculates days since given date
    if pd.isna(date_str): return np.nan
    try:
        d = datetime.strptime(str(date_str), "%Y-%m-%d")
        return (datetime.now() - d).days
    except:
        return 0

# New features
df['amenities_count'] = df['amenities'].apply(count_items)
df['host_days_active'] = df['host_since'].apply(days_since)
df['days_since_first_review'] = df['first_review'].apply(days_since)
df['days_since_last_review'] = df['last_review'].apply(days_since)

# Binary conversion: NaN -> False/0
binary_cols = ['host_is_superhost', 'host_identity_verified', 'instant_bookable']
for col in binary_cols:
    df[col] = df[col].map({'t': 1, 'f': 0}).fillna(0)

## Feature Selection

In [ ]:
TARGET = 'price'

NUMERICAL_STANDARD_FEATURES = [
    'latitude', 'longitude',
    'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'amenities_count',
    'host_days_active', 'days_since_first_review', 'days_since_last_review',
    'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm',
    'calculated_host_listings_count',
    'availability_30', 'availability_60', 'availability_90', 'availability_365',
    'number_of_reviews',
    'host_is_superhost', 'host_identity_verified', 'instant_bookable'
]

NUMERICAL_REVIEW_FEATURES = [
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication', 'review_scores_location',
    'review_scores_value', 'reviews_per_month'
]

CATEGORICAL_FEATURES = [
    'neighbourhood_group_cleansed',
    'neighbourhood_cleansed',
    'room_type',
    'property_type',
    'host_response_time'
]

x = df[NUMERICAL_STANDARD_FEATURES + NUMERICAL_REVIEW_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET].values

## Preprocessor Definition

In [ ]:
# Review pipeline: impute 0 -> StandardScaler
review_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

# Numeric pipeline: impute median -> StandardScaler
standard_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: impute 'unknown' -> OneHot
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('reviews', review_transformer, NUMERICAL_REVIEW_FEATURES),
        ('num', standard_numeric_transformer, NUMERICAL_STANDARD_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

## Transformation Pipeline

In [ ]:
# Apply transformations
x_processed = preprocessor.fit_transform(x)

# Reconstruct DataFrame with new column names
feature_names = preprocessor.get_feature_names_out()
df_processed = pd.DataFrame(x_processed, columns=feature_names)
df_processed[TARGET] = y

print(f"Processed shape: {df_processed.shape}")

## Artifacts Export

In [ ]:
# Save processed data
df_processed.to_csv('../../data/madrid/listings_experimental_processed.csv', index=False)
print("Processed dataset saved to 'data/madrid/listings_experimental_processed.csv'")

# Save preprocessor object
os.makedirs('../../artifacts', exist_ok=True)
with open('../../artifacts/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)
print("Preprocessor saved to 'artifacts/preprocessor.pkl'")

# Preprocessed data head
display(df_processed.head())